In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
np.random.seed(42)


In [4]:
tokens = ["I", "love", "AI"]

seq_len = len(tokens)
d_model = 6        # model dimension
num_heads = 3      # number of attention heads
head_dim = d_model // num_heads

assert d_model % num_heads == 0


In [5]:
E = np.array([
    [1.0, 0.0, 0.2, 0.1, 0.0, 0.3],   # I
    [0.8, 0.6, 0.1, 0.2, 0.1, 0.0],   # love
    [0.0, 1.0, 0.3, 0.0, 0.2, 0.4],   # AI
])

print("Embeddings (E):", E.shape)


Embeddings (E): (3, 6)


In [6]:
# Projection Matrices (Q, K, V, O)

W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model)

# Linear Projections

Q = E @ W_Q
K = E @ W_K
V = E @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)



Q shape: (3, 6)
K shape: (3, 6)
V shape: (3, 6)


In [7]:
W_V

array([[-0.036,  1.565, -2.62 ,  0.822,  0.087, -0.299],
       [ 0.092, -1.988, -0.22 ,  0.357,  1.478, -0.518],
       [-0.808, -0.502,  0.915,  0.329, -0.53 ,  0.513],
       [ 0.097,  0.969, -0.702, -0.328, -0.392, -1.464],
       [ 0.296,  0.261,  0.005, -0.235, -1.415, -0.421],
       [-0.343, -0.802, -0.161,  0.404,  1.886,  0.175]])

In [8]:
def split_heads(x):
    return x.reshape(seq_len, num_heads, head_dim)

# Each token now exists in multiple spaces.

Qh = split_heads(Q)
Kh = split_heads(K)
Vh = split_heads(V)

print("Qh shape:", Qh.shape)


Qh shape: (3, 3, 2)


In [9]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

contexts = []

# -- Each head:

# -- produces its own attention matrix (3 × 3)

# -- learns different relationships

for h in range(num_heads):
    Q_h = Qh[:, h, :]      # (seq_len, head_dim)
    K_h = Kh[:, h, :]
    V_h = Vh[:, h, :]

    scores = Q_h @ K_h.T
    scores /= np.sqrt(head_dim)

    attention = softmax(scores, axis=-1)
    context = attention @ V_h

    contexts.append(context)

    print(f"\nHead {h+1} attention:")
    print(attention)



Head 1 attention:
[[0.359 0.35  0.292]
 [0.395 0.369 0.236]
 [0.268 0.28  0.452]]

Head 2 attention:
[[0.521 0.287 0.192]
 [0.546 0.308 0.145]
 [0.225 0.51  0.265]]

Head 3 attention:
[[0.396 0.357 0.247]
 [0.356 0.352 0.292]
 [0.383 0.359 0.258]]


In [10]:
# Concatenate along feature dimension
context_concat = np.concatenate(contexts, axis=-1)

print("\nConcatenated context shape:", context_concat.shape)



Concatenated context shape: (3, 6)


In [11]:
context_concat

array([[-0.173, -0.149, -1.986,  0.852,  0.888, -0.506],
       [-0.171,  0.037, -2.099,  0.868,  0.944, -0.507],
       [-0.183, -0.672, -1.737,  0.787,  0.902, -0.508]])

In [12]:
W_O

array([[ 0.258, -0.074, -1.919, -0.027,  0.06 ,  2.463],
       [-0.192,  0.302, -0.035, -1.169,  1.143,  0.752],
       [ 0.791, -0.909,  1.403, -1.402,  0.587,  2.19 ],
       [-0.991, -0.566,  0.1  , -0.503, -1.551,  0.069],
       [-1.062,  0.474, -0.919,  1.55 , -0.783, -0.322],
       [ 0.814, -1.231,  0.227,  1.307, -1.607,  0.185]])

In [15]:
context_concat @ W_O

array([[-3.786,  2.335, -3.296,  3.249, -2.55 , -5.21 ],
       [-3.986,  2.513, -3.515,  3.267, -2.469, -5.328],
       [-3.443,  1.998, -2.929,  3.563, -2.907, -5.09 ]])

In [13]:
# WO ​∈ R dmodel​×dmodel​

output = context_concat @ W_O

print("\nFinal output shape:", output.shape)
print("\nFinal output:")
print(output)



Final output shape: (3, 6)

Final output:
[[-3.786  2.335 -3.296  3.249 -2.55  -5.21 ]
 [-3.986  2.513 -3.515  3.267 -2.469 -5.328]
 [-3.443  1.998 -2.929  3.563 -2.907 -5.09 ]]
